# Phase 9: Model Explainability with SHAP — Telco Customer Churn

## 1. Executive Summary & Objective
In **Phase 9**, we transition from raw predictive performance to actionable business intelligence by interpreting the champion **Tuned XGBoost** model using **SHAP (SHapley Additive exPlanations)**.

### Objectives:
1. **Global Interpretability**: Identify the top features driving churn predictions across the entire customer cohort.
2. **Directional Attribution (Beeswarm Plot)**: Understand whether high or low values of a feature increase or decrease churn probability.
3. **Local Interpretability (Waterfall Plots)**: Deconstruct individual customer predictions into transparent, additive dollar and service contributions to empower frontline retention agents.

In [1]:
# 2. Environment Setup & Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Ensure compatibility if cv2 is absent
if 'cv2' not in sys.modules:
    sys.modules['cv2'] = None

import shap

# Project modules
sys.path.append('..')
from src.feature_engineering import engineer_features, get_feature_types
from src.explain import compute_shap_values, plot_shap_summary, plot_shap_waterfall

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 3. Data Preparation & Loading Champion Pipeline

In [2]:
data_path = Path('../data/processed/cleaned_churn_data.csv')
df_clean = pd.read_csv(data_path)
df_feat = engineer_features(df_clean)

X = df_feat.drop(columns=['Churn'])
y = df_feat['Churn']

# Load champion pipeline
model_pipe = joblib.load('../models/best_model.pkl')
preprocessor = model_pipe.named_steps['preprocessor']
clf = model_pipe.named_steps['clf'] if 'clf' in model_pipe.named_steps else model_pipe.named_steps['classifier']

# Clean feature names for visualization
raw_feature_names = preprocessor.get_feature_names_out()
clean_feature_names = [n.replace('num__', '').replace('cat__', '') for n in raw_feature_names]

# Preprocess evaluation sample
sample_size = 500
X_sample = X.iloc[:sample_size]
y_sample = y.iloc[:sample_size]
X_proc = preprocessor.transform(X_sample)

print(f"Evaluated sample: {sample_size} customers, {len(clean_feature_names)} features.")
print(f"Champion model type: {type(clf).__name__}")

Evaluated sample: 500 customers, 36 features.
Champion model type: XGBClassifier


## 4. Computing SHAP Values with TreeExplainer

In [3]:
explainer = shap.TreeExplainer(clf)
shap_explanation = explainer(X_proc)
shap_explanation.feature_names = clean_feature_names

# Global mean absolute SHAP values
mean_abs_shap = np.abs(shap_explanation.values).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature': clean_feature_names,
    'Mean_|SHAP|_Attribution': mean_abs_shap
}).sort_values(by='Mean_|SHAP|_Attribution', ascending=False).reset_index(drop=True)

print("=== TOP 10 GLOBAL PREDICTIVE FEATURES (SHAP) ===")
print(importance_df.head(10).to_string(index=False))

=== TOP 10 GLOBAL PREDICTIVE FEATURES (SHAP) ===
                       Feature  Mean_|SHAP|_Attribution
                IsMonthToMonth                 0.772157
                        tenure                 0.400033
   InternetService_Fiber optic                 0.247312
PaymentMethod_Electronic check                 0.185487
               SupportServices                 0.129049
                MonthlyCharges                 0.119400
            InternetService_No                 0.117620
          PaperlessBilling_Yes                 0.112139
           AverageMonthlySpend                 0.100852
             Contract_One year                 0.094589


## 5. Global Feature Attribution Summary (Beeswarm Plot)
The summary plot reveals both the **importance** and the **direction of association**:
- **Red dots** = High feature value
- **Blue dots** = Low feature value
- **Right of center line ($>0$)** = Pushes customer toward Churn
- **Left of center line ($<0$)** = Pulls customer toward Retention

In [4]:
summary_img_path = Path('../reports/figures/shap_summary.png')
plot_shap_summary(shap_explanation, X_proc, feature_names=clean_feature_names, save_path=summary_img_path, max_display=12)
print(f"[x] Saved SHAP summary plot to {summary_img_path.resolve()}")

[x] Saved SHAP summary plot to D:\ML\customer-churn-prediction\reports\figures\shap_summary.png


## 6. Local Explainability: High-Risk Customer Waterfall Case Study
Let's inspect a customer with an extremely high predicted churn risk to understand the specific drivers behind their score.

In [5]:
predicted_probs = model_pipe.predict_proba(X_sample)[:, 1]
high_risk_idx = int(np.argmax(predicted_probs))
customer_churn_prob = predicted_probs[high_risk_idx]

print(f"Customer #{high_risk_idx} Predicted Churn Probability: {customer_churn_prob:.2%}")
print(f"Actual Status: {'Churned (1)' if y_sample.iloc[high_risk_idx] == 1 else 'Retained (0)'}")

# Save waterfall plot
waterfall_img_path = Path('../reports/figures/shap_waterfall.png')
plot_shap_waterfall(shap_explanation[high_risk_idx], save_path=waterfall_img_path, max_display=10)
print(f"[x] Saved SHAP waterfall plot to {waterfall_img_path.resolve()}")

Customer #345 Predicted Churn Probability: 91.94%
Actual Status: Retained (0)
[x] Saved SHAP waterfall plot to D:\ML\customer-churn-prediction\reports\figures\shap_waterfall.png


## 7. Strategic Business Interpretations & Actionable Retention Playbook

| Top Feature | Observed Attribution Pattern | Recommended Retention Action |
| :--- | :--- | :--- |
| **`IsMonthToMonth`** | Strongest single positive attribution. Month-to-month contracts massively elevate churn probability. | **Incentivize Annual Commitment**: Offer a 15–20% discount on 1-year or 2-year contract migration. |
| **`tenure`** | Low tenure (blue points) heavily pushes toward churn; long tenure acts as an anchor. | **First-90-Day Onboarding Program**: Automated check-in emails, welcome calls, and setup assistance. |
| **`InternetService_Fiber optic`** | Fiber optic subscription correlates with elevated churn risk despite premium billing. | **Service Quality & Price Audit**: Investigate speed reliability and provide bundled loyalty credits. |
| **`PaymentMethod_Electronic check`** | High positive attribution for churn. | **Auto-Pay Incentive**: Offer a $5/month billing credit for switching to automatic credit card / ACH payments. |
| **`SupportServices`** | Having multiple support/security services pulls predictions strongly to the left (retention). | **Value Bundle Push**: Bundle free Tech Support and Online Security for 6 months on high-risk accounts. |